In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from utils.plotting_utils import setup_plotting_environment, cm_to_inches
from utils.color_constants import ALL_CELL_TYPES
setup_plotting_environment()

input_data_folder = "../../data/main_screen/"

output_folder = '../../outputs/questionable_genes/'
os.makedirs(output_folder, exist_ok=True)

plot_folder = os.path.join(output_folder, 'plots')
os.makedirs(plot_folder, exist_ok=True)

In [ ]:
# info on cytokines
cytokine_info = pd.read_excel("../../annotations/cytokine_info.xlsx", index_col=0, sheet_name="all_cytokines")
cytokine_info_parse = cytokine_info[cytokine_info["parse_cytokine"]]

# ordering
cytokine_info_order = pd.read_excel("../../annotations/cytokine_info.xlsx", index_col=0, sheet_name="suggested_order")
CYTOKINE_ORDER = list(cytokine_info_order["name"])

In [ ]:
from utils.cell_type_mapping import get_relevant_cell_types
alt_flag = False
wells = ["H7", "H8", "H9", "H10", "H11", "H12"]

relevant_cell_types, relevant_cell_type_conversion, relevant_cell_to_pseudobulk = get_relevant_cell_types(alt_flag)

#### Load original DEGs

In [ ]:
deg_processed_path = os.path.join(input_data_folder, 'edger_results_main.parquet')
edger_results = pd.read_parquet(deg_processed_path)

In [ ]:
from utils.processing_utils import create_edger_val_df

edger_df_padj = create_edger_val_df(edger_results, "adj_p_value")
edger_df_log2FC = create_edger_val_df(edger_results, "log_fc")

#### Load per-well DEGs

In [ ]:
from glob import glob

per_well_path = "/Volumes/Extreme Pro/parse_data/fig_1/06_degs_per_pbs_well"
all_stats = []

for cell_type in ALL_CELL_TYPES:
    raw_cell_type = relevant_cell_to_pseudobulk[cell_type]
    files = sorted(glob(os.path.join(per_well_path, f"exp_1_DEGs_{raw_cell_type}_pbs-*.csv")))
    if not files:
        raise ValueError(f"No per-well results found for {cell_type}.")

    dfs = []
    for file in files:
        df = pd.read_csv(file, usecols=['variable', 'cytokine', 'log_fc'])
        df = df[df['cytokine'].isin(CYTOKINE_ORDER)]
        dfs.append(df)

    df = pd.concat(dfs, ignore_index=True)
    stats = df.groupby(['cytokine', 'variable'])['log_fc'].agg(fc_mean='mean', fc_std='std', n_wells='count').reset_index()
    stats = stats.rename(columns={'variable': 'gene'})
    stats['cell_type'] = cell_type
    all_stats.append(stats)

fc_mean_std = pd.concat(all_stats, ignore_index=True)
fc_mean_std = fc_mean_std[['cell_type', 'cytokine', 'gene', 'fc_mean', 'fc_std', 'n_wells']]
fc_mean_std = fc_mean_std.dropna(subset=['fc_mean', 'fc_std']).copy()

### Find consistent outlier genes

In [ ]:
constant_add = 0.25
fc_mean_std['ratio'] = fc_mean_std['fc_std'] / (abs(fc_mean_std['fc_mean']) + constant_add)
fc_mean_std['high_ratio'] = fc_mean_std['ratio'] > 1

In [ ]:
fc_mean_std = fc_mean_std[fc_mean_std['cell_type'].isin(ALL_CELL_TYPES)]
high_counts_gene = (fc_mean_std[fc_mean_std['high_ratio']]
                   .groupby(['cell_type', 'gene'])
                   .size()
                   .reindex(pd.MultiIndex.from_product([fc_mean_std['cell_type'].unique(), 
                                                      fc_mean_std['gene'].unique()], 
                                                     names=['cell_type', 'gene']), 
                           fill_value=0)
                   .sort_values(ascending=False))
high_counts_gene = high_counts_gene.reset_index()
high_counts_gene.columns = ["cell_type", "gene", "count"]

In [ ]:
plt.figure(figsize=cm_to_inches((7, 4)))
plt.hist2d((abs(fc_mean_std['fc_mean'])+constant_add).values.flatten(), 
           fc_mean_std['fc_std'].values.flatten(),
           bins=100, 
           cmap='viridis',
           rasterized=True,
           norm=matplotlib.colors.LogNorm())
total_fraction = fc_mean_std['high_ratio'].sum() / len(fc_mean_std)
plt.plot([0, 10], [0, 10], color="red", linestyle="--", label=f"high var. threshold\n(total fraction: {total_fraction:.2%})")
plt.colorbar(label='log(Count)')
plt.legend(loc="upper right")
plt.xlabel(f"|Log2FC| + {constant_add}")
plt.ylabel(f"$\sigma$(log2FC)")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"fc_mean_std_2dhist.svg"), dpi=300)

In [ ]:
high_counts_gene_across = high_counts_gene.groupby('gene')["count"].sum()
plt.figure(figsize=cm_to_inches((7, 4)))
bins = np.arange(-0.5, max(high_counts_gene_across)+1.5, 1)
vals = high_counts_gene_across.value_counts().sort_index()
plt.bar(vals.index, vals.values)

#plt.axvline(x=high_counts_gene_across['7SK'], color='darkred', linestyle='--', label='7SK')
#plt.text(high_counts_gene_across['7SK']+25, 1000, '7SK', fontsize=8, color='darkred')
plt.yscale('log')
plt.xlabel("per-gene count above threshold")
plt.xlim(-50, 800)
plt.ylim(0.5, 5000)
plt.ylabel("genes")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"high_counts_gene_across.svg"), dpi=300)

In [ ]:
plt.figure(figsize=cm_to_inches((7, 4)))
plt.hist(high_counts_gene['count'], bins=np.arange(-0.5, max(high_counts_gene['count'])+1.5, 1), bottom=1e-1)
fraction = (high_counts_gene['count'] >= 10).sum() / len(high_counts_gene)
plt.text(12, 5000, f'threshold for high variance in cell type\n(total fraction: {fraction:.2%})', fontsize=6, color='red')
plt.axvline(x=10.5, color='red', linestyle='--', label='per cell type threshold')
plt.yscale('log')
plt.xlabel("count above threshold per (cell type, gene)")
plt.ylabel("genes")
plt.ylim(0.5, 1e6)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"high_counts_gene_per_cell.svg"), dpi=300)

In [ ]:
high_counts_gene["high"] = high_counts_gene['count'] >= 10
high_cross_cell_types = high_counts_gene[high_counts_gene['high']]['gene'].value_counts().reindex(high_counts_gene['gene'].unique(), fill_value=0)

In [ ]:
plt.figure(figsize=cm_to_inches((7, 4)))
# Create bins for each possible value
bins = np.arange(-0.5, high_cross_cell_types.max() + 1.5, 1)
plt.hist(high_cross_cell_types, bins=bins, edgecolor='black', bottom=1e-1)
plt.xticks(range(high_cross_cell_types.max() + 1))
plt.axvline(x=4.5, color='red', linestyle='--')
fraction = (high_cross_cell_types > 4.5).sum() / len(high_cross_cell_types)
plt.text(5, 1000, f'threshold for high variance \nacross cell types\n(total fraction: {fraction:.2%})', fontsize=6, color='red')
plt.yscale('log')
plt.ylim(1, 1e5)
plt.xlabel("number of cell types with high variance")
plt.ylabel("genes")
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"high_counts_gene_across_cell_types.svg"), dpi=300)

In [ ]:
questionable_genes = pd.Series(high_cross_cell_types[high_cross_cell_types >= 5].index)
questionable_genes

In [ ]:
questionable_genes.to_csv(os.path.join(input_data_folder, 'questionable_genes.csv'), index=False)

In [ ]:
questionable_genes

In [ ]:
questionable_genes_old = pd.read_csv(os.path.join(input_data_folder, 'questionable_genes_soeren.csv'))

In [ ]:
set(questionable_genes_old['gene']) - (set(questionable_genes))

In [ ]:
(set(questionable_genes)) - set(questionable_genes_old['gene']) 

## Check impact on lfc plots

In [ ]:
edger_df_fc_mean = fc_mean_std.pivot(index='gene', columns=['cell_type', 'cytokine'], values='fc_mean')
edger_df_fc_std = fc_mean_std.pivot(index='gene', columns=['cell_type', 'cytokine'], values='fc_std')

In [ ]:
cytokine = "4-1BBL"
cell_type = "B cell"
x = edger_df_log2FC.loc[:, (cell_type, cytokine)]
y = edger_df_fc_mean.loc[:, (cell_type, cytokine)]

x_genes = x.dropna().index
y_genes = y.dropna().index

genes = x_genes.intersection(y_genes)
x = x.loc[genes]
y = y.loc[genes]
y_error = edger_df_fc_std.loc[genes, (cell_type, cytokine)]
questionable_filter = [g for g in questionable_genes.values if g in genes]
other_filter = [g for g in genes if g not in questionable_filter]

plt.figure(figsize=cm_to_inches((7, 7)))
plt.xlabel("all wells log2fc")
plt.ylabel("mean across wells log2fc")
plt.xlim(-2, 2)
plt.ylim(-2, 2)
plt.text(-1.5, 1.5, f"total questionable = {len(questionable_filter)}", fontsize=8)
plt.plot([-4, 4], [-4, 4], color="black")
plt.errorbar(x[other_filter], y[other_filter], yerr=y_error[other_filter], fmt='o', markersize=4, capsize=2, alpha=0.5, rasterized=True)
plt.errorbar(x[questionable_filter], y[questionable_filter], yerr=y_error[questionable_filter], fmt='o', markersize=4, capsize=2, alpha=0.5, color="red", rasterized=True)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"log2fc_vs_mean_log2fc.{format}"), dpi=300)
plt.show()

# consistency filtering (purely exploratory, is applied elsewhere)

In [ ]:
edger_df_padj = edger_df_padj[~edger_df_padj.index.isin(questionable_genes.values)].fillna(1)
edger_df_log2FoldChange = edger_df_log2FC[~edger_df_log2FC.index.isin(questionable_genes.values)].fillna(0)

from glob import glob
well_fc = {}
well_padj = {}
for cell_type in ALL_CELL_TYPES:
    raw_cell_type = relevant_cell_to_pseudobulk[cell_type]
    files = sorted(glob(os.path.join(per_well_path, f"exp_1_DEGs_{raw_cell_type}_pbs-*.csv")))
    if not files:
        raise ValueError(f"No per-well results found for {cell_type}.")

    dfs = []
    for file in files:
        df = pd.read_csv(file, usecols=['variable', 'cytokine', 'pbs_well', 'log_fc', 'adj_p_value'])
        df = df[df['cytokine'].isin(CYTOKINE_ORDER)]
        dfs.append(df)

    df = pd.concat(dfs, ignore_index=True).rename(columns={'variable': 'gene'})
    well_fc[cell_type] = df.pivot(index='gene', columns=['cytokine', 'pbs_well'], values='log_fc')
    well_padj[cell_type] = df.pivot(index='gene', columns=['cytokine', 'pbs_well'], values='adj_p_value')

columns = pd.MultiIndex.from_product([ALL_CELL_TYPES, CYTOKINE_ORDER, wells], names=['cell_type', 'cytokine', 'well'])
edger_df_log2fc_wells = pd.concat(well_fc, axis=1, names=['cell_type']).reindex(columns=columns)
edger_df_padj_wells = pd.concat(well_padj, axis=1, names=['cell_type']).reindex(columns=columns)

In [ ]:
columns = pd.MultiIndex.from_product([ALL_CELL_TYPES, CYTOKINE_ORDER], names=['cell_type', 'cytokine'])
edger_df_log2FC = edger_df_log2FC.reindex(columns=columns)
edger_df_padj = edger_df_padj.reindex(columns=columns)

In [ ]:
edger_df_padj_significant = edger_df_padj < 0.05
edger_df_log2fc_significant = abs(edger_df_log2FC) > 0.25
edger_df_significant = edger_df_padj_significant & edger_df_log2fc_significant

In [ ]:
edger_df_padj_wells_significant = edger_df_padj_wells.fillna(1) < 0.1
edger_df_log2fc_present = abs(edger_df_log2fc_wells.fillna(0)) > 0.25

criterion_padj = edger_df_padj_wells_significant.T.groupby(level=[0,1]).sum().T > 3
criterion_log2fc = edger_df_log2fc_present.T.groupby(level=[0,1]).sum().T > 3
criterion = ((edger_df_padj_wells_significant & edger_df_log2fc_present).T.groupby(level=[0, 1]).sum().T) >= 4

filtered_degs_well_both = criterion & edger_df_significant
filtered_degs_well_padj = criterion_padj & edger_df_significant
filtered_degs_well_log2fc = criterion_log2fc & edger_df_significant

In [ ]:
no_significant_well_both = filtered_degs_well_both.sum(axis=0)
no_significant_well_padj = filtered_degs_well_padj.sum(axis=0)
no_significant_well_log2fc = filtered_degs_well_log2fc.sum(axis=0)
no_significant_all = edger_df_significant.sum(axis=0)

In [ ]:
no_significant = pd.concat([no_significant_all, no_significant_well_both, no_significant_well_padj, no_significant_well_log2fc], axis=1)
no_significant.columns = ["all", "well_both", "well_padj", "well_log2fc"]

no_significant["ratio_both"] = (no_significant["well_both"]+1)/(no_significant["all"]+1)
no_significant["ratio_padj"] = (no_significant["well_padj"]+1)/(no_significant["all"]+1)
no_significant["ratio_log2fc"] = (no_significant["well_log2fc"]+1)/(no_significant["all"]+1)

In [ ]:
n_cols = 3
n_rows = (len(ALL_CELL_TYPES) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(8, 2*n_rows))
axes = axes.flatten()

for idx, cell_type in enumerate(ALL_CELL_TYPES):
    cell_data = no_significant.loc[cell_type]
    axes[idx].scatter(cell_data["all"], cell_data["well_both"], s=5, label="both")
    axes[idx].scatter(cell_data["all"], cell_data["well_log2fc"], s=5, label="log2fc only")

    axes[idx].plot(cell_data["all"], cell_data["all"], color="black", linestyle="--")
    axes[idx].set_title(f"{cell_type}")
    axes[idx].set_xlabel("All wells")
    axes[idx].set_ylabel("Consistent wells")
    axes[idx].legend(loc="upper left")

# Hide empty subplots
for idx in range(len(ALL_CELL_TYPES), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"significant_comparison_per_cell_type.png"), dpi=300)
plt.show()